In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt

# Paths
DATA_PATH = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME")
WIENER_INPUT_DIR = DATA_PATH / "wiener_filtered_audio_pain"

# Perturbation config
PERTURBATIONS = {
    "intensity": {
        "output_dir": DATA_PATH / "wiener_intensity_perturbations_pain",
        "value_key": "gain_db",
        "levels": {
            "-6dB": -6,
            "-3dB": -3,
            "+3dB": 3,
            "+6dB": 6,
        },
    },
    "gaussian_noise": {
        "output_dir": DATA_PATH / "wiener_gaussian_noise_perturbations_pain",
        "value_key": "noise_std_fraction",
        "levels": {
            "low_gaussian_noise": 0.005,
            "medium_gaussian_noise": 0.02,
            "high_gaussian_noise": 0.05,
            "very_high_gaussian_noise": 0.1,
        },
    },
    "pink_noise": {
        "output_dir": DATA_PATH / "wiener_pink_noise_perturbations_pain",
        "value_key": "noise_std_fraction",
        "levels": {
            "low_pink_noise": 0.005,
            "medium_pink_noise": 0.02,
            "high_pink_noise": 0.05,
            "very_high_pink_noise": 0.1,
        },
    },
    "lowpass": {
        "output_dir": DATA_PATH / "wiener_lowpass_perturbations_pain",
        "value_key": "cutoff_hz",
        "levels": {
            "low_lowpass": 6000,
            "medium_lowpass": 5000,
            "high_lowpass": 4000,
            "very_high_lowpass": 3000,
        },
        "filter_order": 4,
    },
}


def collect_wav_files(root_dir):
    return [
        os.path.join(root, file)
        for root, _, files in os.walk(root_dir)
        for file in files
        if file.endswith(".wav")
    ]


def load_wav_file(file_path):
    return wavfile.read(file_path)


def save_wav_file(file_path, sample_rate, signal):
    signal = np.clip(signal, -32768, 32767).astype(np.int16)
    wavfile.write(file_path, sample_rate, signal)


def make_output_path(input_path, input_root, output_root, perturbation_name):
    relative_path = os.path.relpath(input_path, input_root)
    output_path = os.path.join(output_root, perturbation_name, relative_path)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    return output_path


def generate_pink_noise(n):
    white = np.random.randn(n)
    pink_fft = np.fft.rfft(white)
    freqs = np.fft.rfftfreq(n)

    if len(freqs) > 1:
        freqs[0] = freqs[1]

    pink_fft = pink_fft / np.sqrt(freqs)
    pink = np.fft.irfft(pink_fft, n=n)
    return pink


def apply_lowpass_filter(signal, sample_rate, cutoff_hz, filter_order=4):
    signal = signal.astype(np.float32)
    nyquist = sample_rate / 2

    if cutoff_hz >= nyquist:
        raise ValueError(
            f"cutoff_hz ({cutoff_hz}) must be smaller than Nyquist frequency ({nyquist} Hz)"
        )

    sos = butter(
        N=filter_order,
        Wn=cutoff_hz,
        btype="low",
        fs=sample_rate,
        output="sos"
    )

    return sosfiltfilt(sos, signal)


def apply_perturbation(signal, sample_rate, perturbation_type, value, config):
    signal = signal.astype(np.float32)

    if perturbation_type == "intensity":
        factor = 10 ** (value / 20.0)
        return signal * factor

    if perturbation_type == "gaussian_noise":
        noise_std = value * np.std(signal)
        noise = np.random.normal(0.0, noise_std, size=signal.shape)
        return signal + noise

    if perturbation_type == "pink_noise":
        signal_std = np.std(signal)
        pink_noise = generate_pink_noise(len(signal))
        pink_noise = pink_noise / np.std(pink_noise)
        noise_std = value * signal_std
        pink_noise = pink_noise * noise_std
        return signal + pink_noise

    if perturbation_type == "lowpass":
        filter_order = config.get("filter_order", 4)
        return apply_lowpass_filter(
            signal=signal,
            sample_rate=sample_rate,
            cutoff_hz=value,
            filter_order=filter_order
        )

    raise ValueError(f"Unknown perturbation type: {perturbation_type}")


def process_all_perturbations(audio_files, input_root, perturbation_config):
    all_results = {}

    for perturbation_type, config in perturbation_config.items():
        output_dir = config["output_dir"]
        value_key = config["value_key"]
        levels = config["levels"]

        output_dir.mkdir(parents=True, exist_ok=True)
        processed_rows = []

        for input_path in audio_files:
            try:
                sample_rate, signal = load_wav_file(input_path)
                participant_id = os.path.basename(os.path.dirname(input_path))
                filename = os.path.basename(input_path)

                for perturbation_name, value in levels.items():
                    perturbed_signal = apply_perturbation(
                        signal=signal,
                        sample_rate=sample_rate,
                        perturbation_type=perturbation_type,
                        value=value,
                        config=config
                    )

                    output_path = make_output_path(
                        input_path=input_path,
                        input_root=input_root,
                        output_root=output_dir,
                        perturbation_name=perturbation_name
                    )

                    save_wav_file(output_path, sample_rate, perturbed_signal)

                    row = {
                        "participant_id": participant_id,
                        "filename": filename,
                        "original_wiener_file_path": input_path,
                        "perturbation_type": perturbation_type,
                        "perturbation": perturbation_name,
                        value_key: value,
                        "perturbed_file_path": output_path,
                    }

                    if perturbation_type == "lowpass":
                        row["filter_order"] = config.get("filter_order", 4)

                    processed_rows.append(row)

            except Exception as e:
                print(f"Error processing {input_path}: {e}")

        all_results[perturbation_type] = pd.DataFrame(processed_rows)

    return all_results


# Run
wiener_audio_files = collect_wav_files(WIENER_INPUT_DIR)
print(f"Number of Wiener-filtered audio files: {len(wiener_audio_files)}")

results = process_all_perturbations(
    audio_files=wiener_audio_files,
    input_root=WIENER_INPUT_DIR,
    perturbation_config=PERTURBATIONS,
)

df_intensity = results["intensity"]
df_gaussian = results["gaussian_noise"]
df_pink = results["pink_noise"]
df_lowpass = results["lowpass"]

print("Finished processing all perturbations.")

Number of Wiener-filtered audio files: 1055
Finished processing all perturbations.
